# freqgen — SPAI Evasion Experiment (Kaggle)

**Kaggle version** — P100/T4 GPU, 30h/week free, separate quota from Colab.

Differences from Colab v2:
- Paths: `/kaggle/working/` instead of `/content/`
- Synthbuster: downloaded directly to `/kaggle/working/` (no Drive needed;
  Kaggle sessions persist during the session and output is saveable)
- Runtime: GPU accelerator enabled in Settings → Accelerator → GPU

**Enable GPU:** Settings (gear icon, top right) → Accelerator → GPU P100 or T4

## 1. Install SPAI

In [ ]:
import os, subprocess, sys

# Clone SPAI
if not os.path.exists('/kaggle/working/spai'):
    !git clone https://github.com/mever-team/spai.git /kaggle/working/spai -q
%cd /kaggle/working/spai

# Install PyTorch (Kaggle usually has it; reinstall correct CUDA version)
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q

# Install SPAI requirements
!pip install -r requirements.txt filetype -q

import numpy as np, torch
print(f'numpy {np.__version__}  torch {torch.__version__}  cuda={torch.cuda.is_available()}')
print('SPAI install OK')


## 2. Download SPAI weights

In [ ]:
import os
os.makedirs('/kaggle/working/spai/weights', exist_ok=True)
if not os.path.exists('/kaggle/working/spai/weights/spai.pth'):
    !pip install gdown -q
    !gdown 1vvXmZqs6TVJdj8iF1oJ4L_fcgdQrp_YI -O /kaggle/working/spai/weights/spai.pth
sz = os.path.getsize('/kaggle/working/spai/weights/spai.pth') // 1_000_000
print(f'Weights: {sz} MB  (expect ~934)')


## 3. Download data

COCO val2017 (~778 MB) for real images. Synthbuster SD1.4 (~12 GB) for fakes.
Kaggle output directory `/kaggle/working/` has ~20 GB space.

In [ ]:
import os, glob
BASE = '/kaggle/working'

# COCO val2017 (real)
COCO = f'{BASE}/coco_val2017'
if not os.path.exists(COCO) or len(os.listdir(COCO)) < 100:
    print('Downloading COCO val2017 (~778 MB)...')
    !wget -q -c 'http://images.cocodataset.org/zips/val2017.zip' -O /tmp/cv.zip
    !unzip -q /tmp/cv.zip -d {BASE}
    os.rename(f'{BASE}/val2017', COCO)
    os.remove('/tmp/cv.zip')
print(f'Real: {len(os.listdir(COCO))} COCO images')

# Synthbuster SD1.4 (fakes)
SYNTH = f'{BASE}/synthbuster/stable-diffusion-1-4'
if not os.path.exists(SYNTH) or len(os.listdir(SYNTH)) < 100:
    print('Downloading Synthbuster SD1.4 (~12 GB, ~15 min)...')
    !wget -L -c 'https://zenodo.org/records/10066460/files/synthbuster.zip' -O /tmp/sb.zip
    sz_mb = os.path.getsize('/tmp/sb.zip') // 1_000_000
    print(f'Downloaded: {sz_mb} MB')
    if sz_mb < 100:
        raise RuntimeError(f'Download failed ({sz_mb} MB).')
    os.makedirs(os.path.dirname(SYNTH), exist_ok=True)
    !unzip -q /tmp/sb.zip 'synthbuster/stable-diffusion-1-4/*' -d {BASE}
    os.remove('/tmp/sb.zip')
    print(f'Synthbuster extracted: {len(os.listdir(SYNTH))} images')
else:
    print(f'Synthbuster already present: {len(os.listdir(SYNTH))} images')

FAKE_DIR = SYNTH
print(f'Fake: {len(glob.glob(f"{FAKE_DIR}/*.png"))} SD1.4 images ready')


## 4. Spectral matching attack

Rewrite each fake's Fourier magnitude to match the real spectral target.
Phase (structure/content) is preserved.

In [ ]:
import numpy as np, glob, os
from PIL import Image
from tqdm.notebook import tqdm

SIZE = 256
BASE = '/kaggle/working'
COCO = f'{BASE}/coco_val2017'
FAKE_DIR = f'{BASE}/synthbuster/stable-diffusion-1-4'

def load_gray(p):
    return np.array(Image.open(p).convert('L').resize((SIZE,SIZE)), dtype=np.float64)

def radial_profile(ch):
    F = np.fft.fftshift(np.fft.fft2(ch)); mag = np.abs(F)
    cy, cx = ch.shape[0]//2, ch.shape[1]//2
    y, x = np.ogrid[:ch.shape[0], :ch.shape[1]]
    r = np.round(np.sqrt((y-cy)**2+(x-cx)**2)).astype(int)
    mr = min(ch.shape)//2
    t = np.bincount(r.ravel(), weights=mag.ravel())
    c = np.bincount(r.ravel())
    return t[:mr] / np.maximum(c[:mr], 1)

def spectral_match(img, target, gain_clip=(0.1, 12.0)):
    F = np.fft.fftshift(np.fft.fft2(img))
    cy, cx = img.shape[0]//2, img.shape[1]//2
    y, x = np.ogrid[:img.shape[0], :img.shape[1]]
    r = np.round(np.sqrt((y-cy)**2+(x-cx)**2)).astype(int)
    mr = len(target)
    gain = np.clip(target/(radial_profile(img)+1e-12), *gain_clip)
    gain[0] = 1.0
    gmap = gain[np.clip(r,0,mr-1)]; gmap[r>=mr]=1.0; gmap[r==0]=1.0
    return np.clip(np.fft.ifft2(np.fft.ifftshift(F*gmap)).real, 0, 255)

N = 30
real_paths = sorted(glob.glob(f'{COCO}/*.jpg'))[:N]
fake_paths = sorted(glob.glob(f'{FAKE_DIR}/*.png'))[:N]
print(f'real: {len(real_paths)}  fake: {len(fake_paths)}')

print('Building real spectral target...')
target = np.mean([radial_profile(load_gray(p)) for p in tqdm(real_paths)], axis=0)

MATCHED = f'{BASE}/matched_sd14'
os.makedirs(MATCHED, exist_ok=True)
matched_paths = []
print('Running spectral matching attack...')
for p in tqdm(fake_paths):
    m = spectral_match(load_gray(p), target)
    out = f'{MATCHED}/{os.path.basename(p)}'
    Image.fromarray(m.astype(np.uint8)).save(out)
    matched_paths.append(out)
print(f'Saved {len(matched_paths)} matched fakes')

rh = np.mean([radial_profile(load_gray(p))[60:].mean() for p in real_paths])
fh = np.mean([radial_profile(load_gray(p))[60:].mean() for p in fake_paths])
mh = np.mean([radial_profile(load_gray(p))[60:].mean() for p in matched_paths])
print(f'\n=== SPECTRAL GAP ===')
print(f'real={rh:.1f}  fake={fh:.1f}  matched={mh:.1f}')
print(f'Gap real/fake={rh/fh:.2f}x    real/matched={rh/mh:.2f}x')


## 5. Prepare directories for SPAI

In [ ]:
import os, shutil, glob
BASE = '/kaggle/working'

SPAI_IN = f'{BASE}/spai_input'
N = 30
for tag, paths in [
    ('real',    sorted(glob.glob(f'{BASE}/coco_val2017/*.jpg'))[:N]),
    ('fake',    sorted(glob.glob(f'{BASE}/synthbuster/stable-diffusion-1-4/*.png'))[:N]),
    ('matched', sorted(glob.glob(f'{BASE}/matched_sd14/*.png'))[:N]),
]:
    d = f'{SPAI_IN}/{tag}'
    os.makedirs(d, exist_ok=True)
    for p in paths:
        dst = f'{d}/{os.path.basename(p)}'
        if not os.path.exists(dst): shutil.copy2(p, dst)
    print(f'{tag}: {len(os.listdir(d))} images')


## 6. SPAI inference

Run from `/kaggle/working/spai` so `./weights/` and `./configs/` resolve.

In [ ]:
import os
BASE = '/kaggle/working'

for tag in ['real','fake','matched']:
    os.makedirs(f'{BASE}/spai_output/{tag}', exist_ok=True)

%cd /kaggle/working/spai

print('SPAI on REAL images...')
!python -m spai infer \
    --input /kaggle/working/spai_input/real \
    --output /kaggle/working/spai_output/real

print('SPAI on FAKE images...')
!python -m spai infer \
    --input /kaggle/working/spai_input/fake \
    --output /kaggle/working/spai_output/fake

print('SPAI on MATCHED (attacked) fakes...')
!python -m spai infer \
    --input /kaggle/working/spai_input/matched \
    --output /kaggle/working/spai_output/matched

print('SPAI inference done')


## 7. Results — the evasion table

In [ ]:
import pandas as pd, glob, numpy as np
BASE = '/kaggle/working'

def load_scores(tag):
    csvs = glob.glob(f'{BASE}/spai_output/{tag}/**/*.csv', recursive=True)
    if not csvs: raise FileNotFoundError(f'No SPAI output for {tag}')
    df = pd.read_csv(csvs[0])
    print(f'{tag}: {len(df)} rows, columns: {list(df.columns)}')
    return df

res_real    = load_scores('real')
res_fake    = load_scores('fake')
res_matched = load_scores('matched')

score_col = 'spai'  # confirmed column name from previous run
print(f'Score range fake: {res_fake[score_col].min():.3f} - {res_fake[score_col].max():.3f}')

fake_det    = (res_fake[score_col]    >= 0.5).mean()
matched_det = (res_matched[score_col] >= 0.5).mean()
real_fp     = (res_real[score_col]    >= 0.5).mean()

print()
print('='*55)
print('freqgen — SPAI Evasion Table (Synthbuster SD1.4 + COCO)')
print('='*55)
print(f'Spectral gap  real/fake={rh/fh:.1f}x  real/matched={rh/mh:.2f}x')
print(f'SPAI detects raw fakes:       {fake_det:.0%}')
print(f'SPAI detects matched fakes:   {matched_det:.0%}')
print(f'SPAI false-positives real:    {real_fp:.0%}')
print(f'Evasion rate (attack):        {1-matched_det:.0%}')
print('='*55)
if 1-matched_det > 0.5:
    print('RESULT: Attack EVADES SPAI -> gap found in CVPR 2025 SOTA')
else:
    print('RESULT: SPAI survives attack -> learned detectors are robust')

print()
print('Score distributions (real SD1.4 data):')
print(f'  Real    mean={res_real[score_col].mean():.3f}  std={res_real[score_col].std():.3f}')
print(f'  Fake    mean={res_fake[score_col].mean():.3f}  std={res_fake[score_col].std():.3f}')
print(f'  Matched mean={res_matched[score_col].mean():.3f}  std={res_matched[score_col].std():.3f}')
